# mech-sim Colab Workflow

This notebook rebuilds and reruns `crates/mech-sim` from scratch and then regenerates figures, a PDF report, and a downloadable zip bundle.

## Why this notebook exists

The paper argues that gigawatt-class terrestrial legged vehicles are constrained by an energy-power-bandwidth incompatibility:

- a continuous source can support large average power but not the fastest maneuver transients,
- a pulse layer can satisfy burst demand but only if it can later recharge,
- a local actuation layer needs even faster short-horizon buffering and power routing,
- thermal rejection ultimately bounds what can be repeated or sustained.

The original paper did not ship a reduced-order validation layer showing those claims inside a deterministic simulation scaffold. `mech-sim` fills that gap.

## Reduced-order model used here

The simulator evolves:

- `Ep(t)`: pulse-layer stored energy,
- `T(t)`: aggregate thermal state,
- `y(t)`: reduced mechanical output,
- `v(t)`: reduced output velocity,
- `Elocal_i(t)`: four limb-local energy buffers.

The architecture-level equations are:

```text
dEp/dt = eta_c * Pc - Ptransfer_total - Pl(Ep, T)
Ct * dT/dt = Qg(Pdelivered, Ptransfer_total, Pl) - Qr(T)
M * y_ddot + D(T) * y_dot + K(y, T) * y = G(Ep, T) * u + d(t)
dElocal_i/dt = Ptransfer_i - Pdraw_i - Plocal_loss
```

The goal is not full vehicle fidelity. The goal is a simulation-ready, reproducible, inspectable reduced-order proof-of-life model that tests whether the paper's architecture is numerically coherent.

## What this validation does and does not provide

What it does provide:

- pulse discharge vs recharge curves,
- thermal rise and rejection under sustained or repeated demand,
- actuator demand vs pulse-layer depletion,
- local limb buffer depletion and recovery,
- explicit machine-readable breach events,
- parameter sweeps across power, storage, rejection, and demand.

What it does not provide:

- multibody leg dynamics,
- terrain contact fidelity,
- high-fidelity hydraulic servo dynamics,
- nuclear primary-loop transients,
- structural load or fatigue predictions.


In [ ]:
import json
import os
import pathlib
import shutil
import subprocess
import sys
import zipfile

def run(cmd, cwd=None, env=None):
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

run(['apt-get', 'update'])
run(['apt-get', 'install', '-y', 'git', 'curl', 'build-essential', 'pkg-config', 'libfontconfig1-dev', 'libfreetype6-dev'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'matplotlib', 'numpy'])

cargo_bin = pathlib.Path.home() / '.cargo' / 'bin'
if shutil.which('cargo') is None:
    run(['bash', '-lc', 'curl https://sh.rustup.rs -sSf | sh -s -- -y'])
os.environ['PATH'] = f"{cargo_bin}:{os.environ['PATH']}"
run(['cargo', '--version'])
run(['rustc', '--version'])

repo_url = os.environ.get('MECH_REPO_URL', 'https://github.com/infinityabundance/mech.git')
workspace = pathlib.Path('/content/mech')
if workspace.exists() and not (workspace / '.git').exists():
    shutil.rmtree(workspace)
if not workspace.exists():
    run(['git', 'clone', repo_url, str(workspace)])

output_root = workspace / 'output-mech-sim'
shutil.rmtree(output_root, ignore_errors=True)
artifact_dir = workspace / 'crates' / 'mech-sim' / 'notebooks' / 'generated-artifacts'
shutil.rmtree(artifact_dir, ignore_errors=True)
artifact_dir.mkdir(parents=True, exist_ok=True)

workspace

In [ ]:
def run_and_capture(args):
    before = {p.name for p in output_root.iterdir()} if output_root.exists() else set()
    run(['cargo', 'run', '-p', 'mech-sim', '--', *args], cwd=workspace)
    after = sorted([p for p in output_root.iterdir() if p.name not in before])
    if len(after) != 1:
        raise RuntimeError(f'Expected exactly one new output directory, found: {after}')
    return after[0]

run(['cargo', 'build', '-p', 'mech-sim'], cwd=workspace)

burst_dir = run_and_capture(['scenario', 'burst'])
hover_dir = run_and_capture(['scenario', 'hover'])
stress_dir = run_and_capture(['scenario', 'stress'])
sweep_dir = run_and_capture(['sweep', 'baseline'])

run_dirs = {
    'burst': burst_dir,
    'hover': hover_dir,
    'stress': stress_dir,
    'sweep': sweep_dir,
}
run_dirs

## Single-run figures

The next cell reloads the Rust-generated artifacts with pandas and recreates the core figures from scratch. The burst case shows pulse discharge and recharge. The hover case isolates thermal accumulation and the thermal threshold event. The stress case exposes limb-local buffer behavior.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

burst_ts = pd.read_csv(burst_dir / 'time_series.csv')
hover_ts = pd.read_csv(hover_dir / 'time_series.csv')
stress_limb = pd.read_csv(stress_dir / 'limb_buffers.csv')
burst_summary = json.loads((burst_dir / 'summary.json').read_text())
hover_summary = json.loads((hover_dir / 'summary.json').read_text())
stress_summary = json.loads((stress_dir / 'summary.json').read_text())

plt.style.use('seaborn-v0_8-whitegrid')

def save(fig, name):
    path = artifact_dir / name
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return path

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(burst_ts['time_s'], burst_ts['ep_gj'], lw=2.5, color='#1f77b4')
ax.set_title('Burst Scenario: Pulse-Layer Energy vs Time')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Ep [GJ]')
burst_ep_path = save(fig, 'burst_ep_vs_time.png')

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(burst_ts['time_s'], burst_ts['requested_actuator_power_w'] / 1e6, lw=2.2, label='Requested', color='#ff7f0e')
ax.plot(burst_ts['time_s'], burst_ts['delivered_actuator_power_w'] / 1e6, lw=2.2, label='Delivered', color='#2ca02c')
ax.set_title('Burst Scenario: Actuator Power Draw')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Power [MW]')
ax.legend()
burst_power_path = save(fig, 'burst_power_vs_time.png')

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(burst_ts['time_s'], burst_ts['y_m'], lw=2.5, color='#9467bd')
ax.set_title('Burst Scenario: Reduced Mechanical Output y(t)')
ax.set_xlabel('Time [s]')
ax.set_ylabel('y [m]')
burst_y_path = save(fig, 'burst_y_vs_time.png')

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(hover_ts['time_s'], hover_ts['temperature_c'], lw=2.5, color='#d62728')
ax.axhline(hover_summary['peak_temperature_k'] - 273.15, color='#444444', ls='--', lw=1.2, label='Peak T')
ax.axhline(63.0, color='#000000', ls=':', lw=1.2, label='Thermal limit scale')
ax.set_title('Hover Scenario: Aggregate Thermal Rise')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Temperature [C]')
ax.legend()
hover_temp_path = save(fig, 'hover_temperature_vs_time.png')

fig, ax = plt.subplots(figsize=(10, 4.8))
for limb, group in stress_limb.groupby('limb'):
    ax.plot(group['time_s'], group['buffer_energy_mj'], lw=2.0, label=limb)
ax.set_title('Stress Scenario: Local Limb Buffer Trajectories')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Buffer Energy [MJ]')
ax.legend(ncol=2)
stress_limb_path = save(fig, 'stress_limb_buffers.png')

fig, ax = plt.subplots(figsize=(6.4, 5.4))
ax.plot(burst_ts['ep_gj'], burst_ts['temperature_c'], lw=2.2, color='#008c95')
ax.set_title('Burst Scenario: Ep-T Phase Portrait')
ax.set_xlabel('Ep [GJ]')
ax.set_ylabel('Temperature [C]')
phase_path = save(fig, 'burst_phase_ep_t.png')

single_run_report = {
    'burst_summary': burst_summary,
    'hover_summary': hover_summary,
    'stress_summary': stress_summary,
}
(artifact_dir / 'single_run_report.json').write_text(json.dumps(single_run_report, indent=2))
single_run_report

## Sweep figures

The sweep reloads `sweep_summary.csv` and regenerates the key publication-style trend plots. These are the sweep-level proof-of-life artifacts that connect architecture parameters to recharge time, thermal peak, and threshold exposure.


In [ ]:
sweep_summary = pd.read_csv(sweep_dir / 'sweep_summary.csv')

def group(group_name):
    return sweep_summary[sweep_summary['group'] == group_name].copy().sort_values(by=list(sweep_summary.columns))

recharge_pc = sweep_summary[sweep_summary['group'] == 'recharge_pc'].copy().sort_values('continuous_power_mw')
thermal_rejection = sweep_summary[sweep_summary['group'] == 'thermal_rejection'].copy().sort_values('thermal_rejection_mw_per_k')
burst_power = sweep_summary[sweep_summary['group'] == 'burst_power'].copy().sort_values('burst_power_mw')

fig, ax = plt.subplots(figsize=(8.8, 4.8))
recharge_valid = recharge_pc.dropna(subset=['recharge_time_s'])
ax.plot(recharge_valid['continuous_power_mw'], recharge_valid['recharge_time_s'], marker='o', lw=2.2, color='#1f77b4')
ax.set_title('Sweep: Pc vs Recharge Time')
ax.set_xlabel('Continuous Power Pc [MW]')
ax.set_ylabel('Recharge Time [s]')
pc_recharge_path = save(fig, 'sweep_pc_vs_recharge.png')

fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.plot(thermal_rejection['thermal_rejection_mw_per_k'], thermal_rejection['peak_temperature_c'], marker='o', lw=2.2, color='#d62728')
ax.set_title('Sweep: Thermal Rejection vs Peak Temperature')
ax.set_xlabel('Thermal Rejection [MW/K]')
ax.set_ylabel('Peak Temperature [C]')
thermal_rejection_path = save(fig, 'sweep_thermal_rejection_vs_peak_t.png')

fig, ax = plt.subplots(figsize=(8.8, 4.8))
burst_threshold = burst_power.copy()
burst_threshold['time_to_threshold_or_duration'] = burst_threshold['time_to_any_threshold_s'].fillna(90.0)
ax.plot(burst_threshold['burst_power_mw'], burst_threshold['time_to_threshold_or_duration'], marker='o', lw=2.2, color='#ff7f0e')
ax.set_title('Sweep: Burst Power vs Time to First Threshold')
ax.set_xlabel('Burst Power [MW]')
ax.set_ylabel('Time to First Threshold [s]')
burst_threshold_path = save(fig, 'sweep_burst_power_vs_time_to_threshold.png')

fig, ax = plt.subplots(figsize=(8.8, 4.8))
storage = sweep_summary[sweep_summary['group'] == 'pulse_storage'].copy().sort_values('pulse_energy_gj')
ax.plot(storage['pulse_energy_gj'], storage['effective_duty_cycle'], marker='o', lw=2.2, color='#2ca02c')
ax.set_title('Sweep: Pulse Storage vs Effective Duty Cycle')
ax.set_xlabel('Pulse Storage [GJ]')
ax.set_ylabel('Effective Duty Cycle [-]')
storage_path = save(fig, 'sweep_pulse_storage_vs_duty_cycle.png')

sweep_summary.head()

## PDF report and zip bundle

The final cell writes a compact PDF report from the regenerated notebook figures and creates a zip archive containing:

- single-run outputs,
- sweep outputs,
- regenerated notebook figures,
- summary JSON/CSV files,
- the PDF report,
- this notebook file.


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

report_figures = [
    (burst_ep_path, 'Burst scenario pulse-layer energy'),
    (burst_power_path, 'Burst scenario actuator demand and delivery'),
    (burst_y_path, 'Burst scenario reduced mechanical response'),
    (hover_temp_path, 'Hover-equivalent thermal accumulation'),
    (stress_limb_path, 'Stress scenario local limb buffers'),
    (phase_path, 'Burst phase portrait'),
    (pc_recharge_path, 'Sweep: Pc vs recharge time'),
    (thermal_rejection_path, 'Sweep: thermal rejection vs peak temperature'),
    (burst_threshold_path, 'Sweep: burst power vs time to threshold'),
    (storage_path, 'Sweep: pulse storage vs duty cycle'),
]

pdf_path = artifact_dir / 'mech_sim_report.pdf'
with PdfPages(pdf_path) as pdf:
    fig = plt.figure(figsize=(11, 8.5))
    fig.text(0.05, 0.92, 'mech-sim Proof-of-Life Report', fontsize=22, fontweight='bold')
    fig.text(0.05, 0.84, f'Burst output root: {burst_dir}', fontsize=11)
    fig.text(0.05, 0.80, f'Hover output root: {hover_dir}', fontsize=11)
    fig.text(0.05, 0.76, f'Stress output root: {stress_dir}', fontsize=11)
    fig.text(0.05, 0.72, f'Sweep output root: {sweep_dir}', fontsize=11)
    fig.text(0.05, 0.64, f"Burst min Ep: {burst_summary['min_ep_j'] / 1e9:.3f} GJ", fontsize=12)
    fig.text(0.05, 0.60, f"Recharge case near-60 s refill: {pd.read_csv(sweep_dir / 'sweep_summary.csv').query('case_id == \"recharge_pc_50mw\"')['recharge_time_s'].iloc[0]:.2f} s", fontsize=12)
    fig.text(0.05, 0.56, f"Hover first thermal breach: {hover_summary['first_thermal_breach_s']:.2f} s", fontsize=12)
    fig.text(0.05, 0.52, f"Stress saturation count: {stress_summary['saturation_count']}", fontsize=12)
    fig.text(0.05, 0.44, 'This report is a reduced-order architecture validation artifact. It supports pulse-power, recharge, thermal, and local-buffer claims, but it is not a multibody or terrain-contact simulator.', fontsize=12, wrap=True)
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    for image_path, title in report_figures:
        fig, ax = plt.subplots(figsize=(11, 8.5))
        ax.imshow(plt.imread(image_path))
        ax.axis('off')
        fig.suptitle(title, fontsize=18)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

zip_path = artifact_dir / 'mech_sim_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root in [burst_dir, hover_dir, stress_dir, sweep_dir, artifact_dir]:
        for path in pathlib.Path(root).rglob('*'):
            if path.is_file() and path != zip_path:
                zf.write(path, arcname=path.relative_to(workspace))
    notebook_path = workspace / 'crates' / 'mech-sim' / 'notebooks' / 'mech_sim_colab.ipynb'
    zf.write(notebook_path, arcname=notebook_path.relative_to(workspace))

print('PDF report:', pdf_path)
print('Zip bundle:', zip_path)
